In [1]:
import copy
import sys
import os

# Add the project root (i.e., the parent of 'experiments/' and 'src_torch/')
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [2]:
import copy
from src.regulariser import MultiRegulariser, L2Regulariser, UnbiasRegulariser, L1Regulariser

import torch 
import numpy as np
from src.data_utils import EmbeddingDatasetExtractor
%load_ext autoreload
%autoreload 2
import sys
from abstract_gradient_training.bounded_models import IntervalBoundedModel

import os
import sklearn
from src.interval_utils import get_balanced_min_acc, _get_min_acc
from torch.utils.data import DataLoader

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)
from src.trainer import IntervalTrainer, FisherTrainer, SimpleTrainer

In [3]:
torch.manual_seed(0)
torch.cuda.manual_seed(0)
torch.backends.cudnn.deterministic = True

In [4]:
def label_map(labels):
    labels_new = np.ones_like(labels)
    labels_new[labels == 2] = 1 # 4 stars + 
    labels_new[labels == 1] = 1 # 3 stars 
    labels_new[labels == 0] = 0 # 0 and 1 stars 
    return labels_new

DATASET_EXTRACTOR = EmbeddingDatasetExtractor(
    "data/multi_twitter", label_map_fn= label_map
)
VALIDATION_RATIO = 0.1
MINIMUM_ITER_FINETUNING = 25000
initial_dataset_name = "english"

In [5]:
X, y = next(iter(DataLoader(DATASET_EXTRACTOR.get_embedding_dataset('gte-large', 'english', balance=False)[1], batch_size=10, shuffle=False)))
print((y==1).sum())
print((y == 0).sum())
print((y == 2).sum())


Dataset not found or cache not used, extracting it now.
Applying map function to labels...
tensor(8)
tensor(2)
tensor(0)


C:\Users\Fasterling Pierre\OneDrive\Bureau\AAAI\CertifiedContinualLearning\src\data_utils.py:357: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:209.)
  return torch.from_numpy(self._features[idx]), torch.tensor(self._labels[idx]).long()


In [6]:
HYPERPARAMETERS = {
    "default" : {
        "min_acc_limit" : 0.76,
        "projection_strategy" : "best_loss",
        "n_certificate_samples" : 2000,
        "rashomon_batchsize" : 1024,
        "n_iters" : 700,
        "obj_alpha" : 0.000, "primal_learning_rate" : 0.02,
        "checkpoint" : 20, "dual_learning_rate" : 0.001,
        "n_epochs" : 40,
        "batchsize" : 64,
        "learning_rate" : 0.0005,
        "weight_decay" : 0.000, "unbias_lambda": 0.0, "l1_lambda": 0.0, "l2_lambda": 0.0
    },
    "mxbai-embed-large-v1" : {
        "min_acc_limit" : 0.73,
        "projection_strategy" : "best_loss",
        "n_certificate_samples" : 2000,
        "rashomon_batchsize" : 1024,
        "n_iters" : 700,
        "obj_alpha" : 0.000, "primal_learning_rate" : 0.002,
        "checkpoint" : 20, "dual_learning_rate" : 0.0001,
        "n_epochs" : 40,
        "batchsize" : 64,
        "learning_rate" : 0.00005,
        "weight_decay" : 0.000, "unbias_lambda": 0.0, "l1_lambda": 0.0, "l2_lambda": 0.0
    },
    
}

In [7]:
device = torch.device("cuda")
def get_model(input_dim: int, seed=0, device="cuda", output_dim=10, head=None):
    """Get a simple CNN model."""
    torch.manual_seed(seed)
    model = torch.nn.Sequential(
        #torch.nn.Linear(input_dim, int(input_dim*0.1)),
        #torch.nn.ReLU(),
        torch.nn.Linear(input_dim, output_dim, bias=True)
    ).to(device)
    if head is not None:
        model.append(head)
    return model

def first_head_train_unconstrained(dataset_name: str, llm_name: str, balance: bool = True):
    dataset_train, dataset_val = DATASET_EXTRACTOR.get_embedding_dataset(
        llm_name, dataset_name, val_ratio=VALIDATION_RATIO, balance=balance, use_cache=True
    )
    X, y = get_batch(dataset_train)
    model = get_model(X.shape[1], output_dim=2)
    n_epochs = max(1, MINIMUM_ITER_FINETUNING//len(dataset_train))
    


    trainer = SimpleTrainer(
        model, 
    )
    
    result_model = trainer.train(
        dataset_train,
        dataset_val,
        n_epochs = n_epochs,
        batchsize = 64,
        learning_rate = 0.0005,
        weight_decay = 0.0,
    )
    return copy.deepcopy(result_model)


def get_batch(dataset, seed=0, device="cuda", batchsize=100, domain_map_fn=None):
    """Get a batch of data from the dataset."""
    torch.manual_seed(seed)
    dl = torch.utils.data.DataLoader(dataset, batch_size=batchsize)
    batch, labels = next(iter(dl))
    batch, labels = batch.to(device), labels.to(device)
    labels = domain_map_fn(labels) if domain_map_fn else labels
    return batch, labels

def evaluate(model: torch.nn.Sequential, dataset_name: str, llm_name: str):
    _, dataset_val = DATASET_EXTRACTOR.get_embedding_dataset(
        llm_name, dataset_name, val_ratio=VALIDATION_RATIO, balance=False, use_cache=True
    )
    X, y = get_batch(dataset_val, batchsize=len(dataset_val))
    model.eval()
    with torch.no_grad():
        preds = torch.argmax(model(X), dim=1)
        accuracy = ((preds == y).sum()/y.shape[0]).item()
        balanced_accuracy = sklearn.metrics.balanced_accuracy_score(
            y.cpu().numpy(), preds.cpu().numpy()
        )
    return accuracy, balanced_accuracy

def certify_balanced_accuracy(bound_model: IntervalBoundedModel, dataset_name: str, llm_name: str): 
    _, dataset_val = DATASET_EXTRACTOR.get_embedding_dataset(
        llm_name, dataset_name, val_ratio=VALIDATION_RATIO, balance=False, use_cache=True
    )
    X, y = get_batch(dataset_val, batchsize=len(dataset_val))
    with torch.no_grad():
        certified_balanced_accuracy = get_balanced_min_acc(
            bound_model, X, y
        )
    return certified_balanced_accuracy
def certify_raw_accuracy(bound_model: IntervalBoundedModel, dataset_name: str, llm_name: str): 
    _, dataset_val = DATASET_EXTRACTOR.get_embedding_dataset(
        llm_name, dataset_name, val_ratio=VALIDATION_RATIO, balance=False, use_cache=True
    )
    X, y = get_batch(dataset_val, batchsize=len(dataset_val))
    with torch.no_grad():
        certified_balanced_accuracy = _get_min_acc(
            bound_model, X, y, soft = False
        )
    return certified_balanced_accuracy.item()
def finetune_head(trainer: IntervalTrainer, dataset_name: str, llm_name: str, balance: bool = True):
    print("-"*10, f"Finetuning {llm_name} on dataset {dataset_name}", "-"*10)
    dataset_train, dataset_val = DATASET_EXTRACTOR.get_embedding_dataset(
        llm_name, dataset_name, val_ratio=VALIDATION_RATIO, balance=balance, use_cache=True
    )
    trainer._last_projection = None 
    model_copy = copy.deepcopy(trainer.model)
    n_epochs = max(1, MINIMUM_ITER_FINETUNING//len(dataset_train))
    trainer.train(
        dataset_train,
        dataset_val,
        epochs = n_epochs,
        batch_size = 64,
        lr = 0.0005,
        weight_decay = 0.000,
        val_freq=1
    )
    result_model = copy.deepcopy(trainer.model)
    trainer.model = model_copy # restore model later 
    return result_model

def first_head_train(dataset_name: str, llm_name: str, balance: bool = True):
    dataset_train, dataset_val = DATASET_EXTRACTOR.get_embedding_dataset(
        llm_name, dataset_name, val_ratio=VALIDATION_RATIO, balance=balance, use_cache=True
    )
    X, y = get_batch(dataset_train)
    model = get_model(X.shape[1], output_dim=2)
    hyperparameters = None 
    if llm_name in HYPERPARAMETERS: 
        hyperparameters = HYPERPARAMETERS[llm_name]
    else:
        hyperparameters = HYPERPARAMETERS["default"]
    
    l2 = L2Regulariser(hyperparameters["l2_lambda"])
    l1 = L1Regulariser(hyperparameters["l1_lambda"])
    unbias = UnbiasRegulariser(hyperparameters["unbias_lambda"])
    regulariser = MultiRegulariser([l2, l1, unbias])


    trainer = IntervalTrainer(
        model, 
        min_acc_limit=hyperparameters["min_acc_limit"],
        projection_strategy = hyperparameters["projection_strategy"],
        n_certificate_samples = hyperparameters["n_certificate_samples"],
        batch_size = hyperparameters["rashomon_batchsize"],
        n_iters = hyperparameters["n_iters"],
        obj_alpha=hyperparameters["obj_alpha"], 
        primal_learning_rate=hyperparameters["primal_learning_rate"],
        checkpoint = hyperparameters["checkpoint"], 
        dual_learning_rate=hyperparameters["dual_learning_rate"],
        task_labels = [(0,1)],
        paradigm="CIL"
    )
    
    trainer.train(
        dataset_train,
        dataset_val,
        epochs = hyperparameters["n_epochs"],
        batch_size = hyperparameters["batchsize"],
        lr = hyperparameters["learning_rate"],
        weight_decay = hyperparameters["weight_decay"],
        regulariser=regulariser,
        val_freq=1
    )

    trainer.compute_rashomon_set(
        dataset_train,
    )
    return trainer, copy.deepcopy(trainer.model)

def print_acc_dict(accuracies_arg):
    for model, accs in accuracies_arg.items():
        print("===> ", model)
        for dataset, acc in accs.items():
            print(f"    |--> Dataset {dataset}")
            print(f"        |--> Accuracies Initial Dataset {initial_dataset_name} (SD/Macro)  : {acc[0]}")
            print(f"        |--> Accuracies Fine-tune dataset (SD/Macro)  : {acc[1]}")

### Reproducibility
To get the same results as in the paper all cells should be run one after the other in the order of this notebook. 

In [8]:
torch.manual_seed(0)
torch.cuda.manual_seed(0)
torch.backends.cudnn.deterministic = True

In [9]:
llm_names = ["mxbai-embed-large-v1", "gte-large", "voyage-3", "gemini", "text-embedding-3-large"]
print("*"*10, f"Initial training of models on {initial_dataset_name}", "*"*10)
trainers = {}
original_models = {}
for llm_name in llm_names:
    print("-"*10, f"Initial training of {llm_name} on {initial_dataset_name}", "-"*10)
    trainers[llm_name], original_models[llm_name] = first_head_train(
        initial_dataset_name, llm_name, balance=True 
    )

********** Initial training of models on english **********
---------- Initial training of mxbai-embed-large-v1 on english ----------
Dataset not found or cache not used, extracting it now.
Applying map function to labels...


Training Epochs: 100%|██████████| 40/40 [00:48<00:00,  1.21s/it, val_loss=0.3327, val_acc=0.8540, proj=None, progress=0.98]


Initial acc constraint violation: -0.1222 (Positive = violated)
Number of model parameters: 2050
Computing Rashomon set with min acc limit: 0.73
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.85,  Min acc soft=0.85


100%|██████████| 700/700 [00:28<00:00, 24.14it/s, size=4.91, obj=0.002, min_soft_acc=0.717]


Final bbox:  Obj=0.00,  Size=4.91,  Min acc hard=0.73,  Min acc soft=0.72
Computing final certificates over 2000 samples
Checkpointed every 20 iterations for a total of 35 checkpoints
Checkpoints sizes: ['0.23', '1.44', '3.03', '3.54', '3.57', '3.66', '3.96', '3.92', '4.11', '4.17', '4.19', '4.23', '4.36', '4.44', '4.38', '4.54', '4.52', '4.54', '4.60', '4.54', '4.60', '4.67', '4.70', '4.62', '4.67', '4.76', '4.70', '4.73', '4.81', '4.78', '4.85', '4.80', '4.85', '4.86', '4.91']
Checkpoint certificates: ['0.84', '0.80', '0.74', '0.73', '0.73', '0.74', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73', '0.73']
----------------------- Finished Computing Rashomon set ------------------------
---------- Initial training of gte-large on english ----------


...

Final bbox:  Obj=0.02,  Size=34.13,  Min acc hard=0.71,  Min acc soft=0.70
Computing final certificates over 2000 samples
Checkpointed every 20 iterations for a total of 35 checkpoints
Checkpoints sizes: ['2.29', '14.45', '29.56', '35.41', '36.35', '36.48', '36.49', '36.47', '36.44', '36.39', '36.33', '36.26', '36.18', '36.10', '36.01', '35.93', '35.84', '35.76', '35.67', '35.58', '35.48', '35.39', '35.30', '35.21', '35.11', '35.02', '34.92', '34.83', '34.73', '34.63', '34.53', '34.43', '34.33', '34.23', '34.13']
Checkpoint certificates: ['0.84', '0.78', '0.72', '0.70', '0.69', '0.69', '0.69', '0.69', '0.69', '0.70', '0.70', '0.70', '0.70', '0.70', '0.70', '0.70', '0.70', '0.70', '0.70', '0.70', '0.70', '0.70', '0.70', '0.70', '0.70', '0.70', '0.70', '0.70', '0.70', '0.70', '0.70', '0.70', '0.70', '0.71', '0.71']
----------------------- Finished Computing Rashomon set ------------------------
---------- Initial training of text-embedding-3-large on english ----------
Dataset not found 

Training Epochs: 100%|██████████| 40/40 [01:01<00:00,  1.53s/it, val_loss=0.2845, val_acc=0.8837, proj=None, progress=0.98]


Initial acc constraint violation: -0.1269 (Positive = violated)
Number of model parameters: 6146
Computing Rashomon set with min acc limit: 0.76
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.89,  Min acc soft=0.89


100%|██████████| 700/700 [00:30<00:00, 22.99it/s, size=145.88, obj=0.024, min_soft_acc=0.675]


Final bbox:  Obj=0.02,  Size=145.88,  Min acc hard=0.72,  Min acc soft=0.71
Computing final certificates over 2000 samples
Checkpointed every 20 iterations for a total of 35 checkpoints
Checkpoints sizes: ['2.29', '14.44', '30.44', '46.44', '62.44', '78.44', '94.44', '110.26', '125.04', '136.93', '143.24', '146.02', '146.86', '147.12', '147.18', '147.20', '147.19', '147.17', '147.13', '147.09', '147.03', '146.96', '146.89', '146.81', '146.72', '146.64', '146.56', '146.47', '146.39', '146.30', '146.22', '146.13', '146.05', '145.96', '145.88']
Checkpoint certificates: ['0.89', '0.88', '0.86', '0.85', '0.83', '0.81', '0.78', '0.76', '0.74', '0.72', '0.72', '0.72', '0.71', '0.71', '0.71', '0.71', '0.71', '0.71', '0.71', '0.71', '0.71', '0.71', '0.71', '0.71', '0.71', '0.71', '0.72', '0.72', '0.72', '0.72', '0.72', '0.72', '0.72', '0.72', '0.72']
----------------------- Finished Computing Rashomon set ------------------------


In [10]:
_, val_data = DATASET_EXTRACTOR.get_embedding_dataset("gemini", "english")
X, y = next(iter(DataLoader(val_data, batch_size=len(val_data))))
print((torch.argmax(original_models["gemini"](X.to(device)), dim=1).cpu() == y).sum()/len(val_data))

tensor(0.8490)


In [11]:
no_finetune_accuracies = {}
finetune_dataset_names = ["arabic", "german","hindi", "portuguese", "italian", "spanish", "french" ]# "civil_comments", "hatemoji", "sbic", "hatecheck"]
for llm_name_to_finetune in llm_names: 
    no_finetune_accuracies[llm_name_to_finetune] = {}
    for finetune_dataset_name in finetune_dataset_names:
        no_finetune_model = original_models[llm_name_to_finetune]
        accs_new_dataset = evaluate(
            no_finetune_model, finetune_dataset_name, llm_name_to_finetune
        )
        accs_initial_dataset = evaluate(
            no_finetune_model, initial_dataset_name, llm_name_to_finetune
        )
        no_finetune_accuracies[llm_name_to_finetune][finetune_dataset_name] = (
            accs_initial_dataset, accs_new_dataset
        )
print("*"*20, "Initial results with no fine-tuning", "*"*20)
print_acc_dict(no_finetune_accuracies)
#9244218468666077
# .6611999869346619 

Dataset not found or cache not used, extracting it now.
Applying map function to labels...
Dataset not found or cache not used, extracting it now.
Applying map function to labels...
Dataset not found or cache not used, extracting it now.
Applying map function to labels...
Dataset not found or cache not used, extracting it now.
Applying map function to labels...
Dataset not found or cache not used, extracting it now.
Applying map function to labels...
Dataset not found or cache not used, extracting it now.
Applying map function to labels...
Dataset not found or cache not used, extracting it now.
Applying map function to labels...
Dataset not found or cache not used, extracting it now.
Applying map function to labels...
Dataset not found or cache not used, extracting it now.
Applying map function to labels...
Dataset not found or cache not used, extracting it now.
Applying map function to labels...
Dataset not found or cache not used, extracting it now.
Applying map function to labels...

In [12]:
accuracies = {}
certificates = {}
for llm_name_to_finetune in llm_names: 
    accuracies[llm_name_to_finetune] = {}
    certificates[llm_name_to_finetune] = {}
    for finetune_dataset_name in finetune_dataset_names:
        current_trainer: IntervalTrainer = trainers[llm_name_to_finetune]
        finetuned_model = finetune_head(
            current_trainer, finetune_dataset_name, llm_name_to_finetune, balance=True
        )
        
        accs_new_dataset = evaluate(
            finetuned_model, finetune_dataset_name, llm_name_to_finetune
        )
        accs_initial_dataset = evaluate(
            finetuned_model, initial_dataset_name, llm_name_to_finetune
        )
        accuracies[llm_name_to_finetune][finetune_dataset_name] = (
            accs_initial_dataset, accs_new_dataset
        )
        
        current_outer_bbox = current_trainer.get_current_bbox()
        certified_balanced_accuracy = certify_balanced_accuracy(
            current_outer_bbox, initial_dataset_name, llm_name_to_finetune
        )
        certificates[llm_name_to_finetune][finetune_dataset_name] = certified_balanced_accuracy
        print(f"=> Done ! Certified balanced accuracy {certified_balanced_accuracy}")
        print(f"    |--> Accuracies on new dataset (SD/Macro) {accs_new_dataset}")
        print(f"    |--> Accuracies on initial dataset {initial_dataset_name} (SD/Macro) {accs_initial_dataset}")
print("-"*20, "Final Results with Certified Fine-Tuning","-"*20)
print_acc_dict(accuracies)

---------- Finetuning mxbai-embed-large-v1 on dataset arabic ----------


Training Epochs: 100%|██████████| 6/6 [00:16<00:00,  2.69s/it, val_loss=0.8139, val_acc=0.5074, proj=15, progress=0.98]


=> Done ! Certified balanced accuracy 0.7326732673267327
    |--> Accuracies on new dataset (SD/Macro) (0.6402640342712402, np.float64(0.5074257425742574))
    |--> Accuracies on initial dataset english (SD/Macro) (0.8184818625450134, np.float64(0.8366336633663367))
---------- Finetuning mxbai-embed-large-v1 on dataset german ----------


Training Epochs: 100%|██████████| 6/6 [00:15<00:00,  2.59s/it, val_loss=0.6101, val_acc=0.6535, proj=9, progress=0.98] 


=> Done ! Certified balanced accuracy 0.7376237623762376
    |--> Accuracies on new dataset (SD/Macro) (0.6600660085678101, np.float64(0.6534653465346535))
    |--> Accuracies on initial dataset english (SD/Macro) (0.8349835276603699, np.float64(0.8465346534653466))
---------- Finetuning mxbai-embed-large-v1 on dataset hindi ----------


...

=> Done ! Certified balanced accuracy 0.8712871287128713
    |--> Accuracies on new dataset (SD/Macro) (0.8547855019569397, np.float64(0.8391089108910892))
    |--> Accuracies on initial dataset english (SD/Macro) (0.8877888321876526, np.float64(0.8836633663366337))
---------- Finetuning text-embedding-3-large on dataset spanish ----------


Training Epochs: 100%|██████████| 6/6 [00:18<00:00,  3.09s/it, val_loss=0.4281, val_acc=0.7698, proj=10, progress=0.98]


=> Done ! Certified balanced accuracy 0.7351485148514851
    |--> Accuracies on new dataset (SD/Macro) (0.7986798882484436, np.float64(0.7698019801980198))
    |--> Accuracies on initial dataset english (SD/Macro) (0.8844884634017944, np.float64(0.8861386138613861))
---------- Finetuning text-embedding-3-large on dataset french ----------


Training Epochs: 100%|██████████| 6/6 [00:22<00:00,  3.72s/it, val_loss=0.4190, val_acc=0.8045, proj=15, progress=0.98]


=> Done ! Certified balanced accuracy 0.7277227722772277
    |--> Accuracies on new dataset (SD/Macro) (0.8283828496932983, np.float64(0.8044554455445545))
    |--> Accuracies on initial dataset english (SD/Macro) (0.8877888321876526, np.float64(0.8910891089108911))
-------------------- Final Results with Certified Fine-Tuning --------------------
===>  mxbai-embed-large-v1
    |--> Dataset arabic
        |--> Accuracies Initial Dataset english (SD/Macro)  : (0.8184818625450134, np.float64(0.8366336633663367))
        |--> Accuracies Fine-tune dataset (SD/Macro)  : (0.6402640342712402, np.float64(0.5074257425742574))
    |--> Dataset german
        |--> Accuracies Initial Dataset english (SD/Macro)  : (0.8349835276603699, np.float64(0.8465346534653466))
        |--> Accuracies Fine-tune dataset (SD/Macro)  : (0.6600660085678101, np.float64(0.6534653465346535))
    |--> Dataset hindi
        |--> Accuracies Initial Dataset english (SD/Macro)  : (0.8151815533638, np.float64(0.83415841584

In [13]:
print("*"*10, "Final Results", "*"*10)
print("Model ", " & ".join([finetune_dataset_name for finetune_dataset_name in finetune_dataset_names]))
for llm_name in llm_names:
    line = llm_name + " & "
    accuracy = accuracies[llm_name]
    accuracy_prev = no_finetune_accuracies[llm_name]
    for finetune_dataset_name in finetune_dataset_names: 
        certificate = certificates[llm_name][finetune_dataset_name]
        acc = accuracy[finetune_dataset_name]
        acc_prev = accuracy_prev[finetune_dataset_name]
        #line += f"{round(acc[0][1], 2)}({round(acc[0][0], 2)})"
        #line += f"/{round(acc[1][1], 2)}({round(acc[1][0], 2)})"
        #line += f"[{round(certificate, 2)}]  & "
        line += ""+str(acc_prev[1][1])+"->"+str(acc[1][1]) + "      "
    print(line)
    
        
        
        # macro_acc(real_acc) / macro_acc(real_acc)

********** Final Results **********
Model  arabic & german & hindi & portuguese & italian & spanish & french
mxbai-embed-large-v1 & 0.5099009900990099->0.5074257425742574      0.6509900990099009->0.6534653465346535      0.5866336633663366->0.6089108910891089      0.5173267326732673->0.5321782178217822      0.6262376237623762->0.6782178217821783      0.5643564356435643->0.6064356435643564      0.7004950495049505->0.7252475247524752      
gte-large & 0.4900990099009901->0.5099009900990099      0.5816831683168316->0.6386138613861386      0.5643564356435643->0.6113861386138614      0.5099009900990099->0.5222772277227723      0.6039603960396039->0.6485148514851484      0.5742574257425743->0.6188118811881188      0.6707920792079208->0.7153465346534653      
voyage-3 & 0.8044554455445545->0.8044554455445545      0.7301980198019802->0.7425742574257426      0.6287128712871287->0.6485148514851485      0.6955445544554455->0.7376237623762376      0.7772277227722773->0.8217821782178218      0.72277

In [14]:
print("*"*10, "Final Results", "*"*10)
print("Model ", " & ".join([finetune_dataset_name for finetune_dataset_name in finetune_dataset_names]))
for llm_name in llm_names:
    line = llm_name + " & "
    accuracy = accuracies[llm_name]
    for finetune_dataset_name in finetune_dataset_names: 
        certificate = certificates[llm_name][finetune_dataset_name]
        acc = accuracy[finetune_dataset_name]
        line += f"{round(acc[0][1], 2)}({round(acc[0][0], 2)})"
        line += f"/{round(acc[1][1], 2)}({round(acc[1][0], 2)})"
        line += f"[{round(certificate, 2)}]  & "
    print(line)
    
        
        
        # macro_acc(real_acc) / macro_acc(real_acc)

********** Final Results **********
Model  arabic & german & hindi & portuguese & italian & spanish & french
mxbai-embed-large-v1 & 0.84(0.82)/0.51(0.64)[0.73]  & 0.85(0.83)/0.65(0.66)[0.74]  & 0.83(0.82)/0.61(0.64)[0.74]  & 0.84(0.83)/0.53(0.66)[0.73]  & 0.84(0.82)/0.68(0.72)[0.73]  & 0.84(0.82)/0.61(0.66)[0.73]  & 0.84(0.83)/0.73(0.76)[0.73]  & 
gte-large & 0.82(0.79)/0.51(0.61)[0.71]  & 0.81(0.81)/0.64(0.69)[0.72]  & 0.83(0.81)/0.61(0.66)[0.72]  & 0.82(0.8)/0.52(0.67)[0.71]  & 0.82(0.8)/0.65(0.71)[0.71]  & 0.82(0.8)/0.62(0.68)[0.71]  & 0.82(0.81)/0.72(0.77)[0.72]  & 
voyage-3 & 0.84(0.82)/0.8(0.82)[0.7]  & 0.83(0.81)/0.74(0.75)[0.7]  & 0.84(0.82)/0.65(0.68)[0.75]  & 0.83(0.81)/0.74(0.79)[0.7]  & 0.83(0.82)/0.82(0.83)[0.7]  & 0.84(0.82)/0.73(0.75)[0.71]  & 0.83(0.82)/0.79(0.81)[0.7]  & 
gemini & 0.83(0.81)/0.5(0.66)[0.69]  & 0.85(0.84)/0.76(0.76)[0.69]  & 0.84(0.81)/0.58(0.62)[0.69]  & 0.84(0.82)/0.69(0.73)[0.72]  & 0.85(0.83)/0.77(0.78)[0.69]  & 0.85(0.83)/0.73(0.75)[0.72]  & 0.85(0

In [18]:
print_acc_dict(no_finetune_accuracies)

===>  mxbai-embed-large-v1
    |--> Dataset arabic
        |--> Accuracies Initial Dataset english (SD/Macro)  : (0.8514851927757263, np.float64(0.8539603960396039))
        |--> Accuracies Fine-tune dataset (SD/Macro)  : (0.6732673645019531, np.float64(0.5099009900990099))
    |--> Dataset german
        |--> Accuracies Initial Dataset english (SD/Macro)  : (0.8514851927757263, np.float64(0.8539603960396039))
        |--> Accuracies Fine-tune dataset (SD/Macro)  : (0.6765676736831665, np.float64(0.6509900990099009))
    |--> Dataset hindi
        |--> Accuracies Initial Dataset english (SD/Macro)  : (0.8514851927757263, np.float64(0.8539603960396039))
        |--> Accuracies Fine-tune dataset (SD/Macro)  : (0.6666666865348816, np.float64(0.5866336633663366))
    |--> Dataset portuguese
        |--> Accuracies Initial Dataset english (SD/Macro)  : (0.8514851927757263, np.float64(0.8539603960396039))
        |--> Accuracies Fine-tune dataset (SD/Macro)  : (0.669966995716095, np.float64(